# DFS System Analysis

Multi-scale statistical analysis of distributive fluvial systems using manual digitisation data.

**Instructions:**
- Change `DATA_PATH` below to analyze a different system file in `../data/`.
- All analysis steps are modular and reusable.

## Import Libraries

In [ ]:
import scienceplots
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from moepy import lowess
plt.style.use(['science','ieee'])

## Configurable Parameters

In [ ]:
# Point this to any system dataset in ../data/
DATA_PATH = "../data/rio_fragua_chorroso_analysis_data.xlsx"  # Change for other systems
SHEETS = {
    "channel_profile": "channel_profile",
    "cross_section_scale": "cross_section_scale",
    "system_scale": "system_scale",
    "domain_scale": "domain_scale",
    "reach_scale": "reach_scale"
}

## Load All Standard Data Sheets

In [ ]:
channel_profile = pd.read_excel(DATA_PATH, sheet_name=SHEETS["channel_profile"])
cross_section = pd.read_excel(DATA_PATH, sheet_name=SHEETS["cross_section_scale"])
system_scale = pd.read_excel(DATA_PATH, sheet_name=SHEETS["system_scale"])
domain_scale = pd.read_excel(DATA_PATH, sheet_name=SHEETS["domain_scale"])
reach_scale = pd.read_excel(DATA_PATH, sheet_name=SHEETS["reach_scale"])

# Quick Data Check
channel_profile.head(), cross_section.head()

## Reusable OLS Regression Function

In [ ]:
def ols_regression(df, y_col, x_col='segment'):
    """
    Run OLS regression and print summary for y_col vs x_col.
    """
    y = df[y_col]
    x = sm.add_constant(df[x_col])
    model = sm.OLS(y, x).fit()
    print(f"OLS regression for {y_col} vs {x_col}:")
    print(model.summary())
    return model

## Channel Profile Analysis (LOWESS Smoothing)

In [ ]:
x = channel_profile['distance_km'].values
y = channel_profile['elevation_m'].values
lowess_model = lowess.Lowess()
lowess_model.fit(x, y)
x_pred = np.linspace(x.min(), x.max(), 30)
y_pred = lowess_model.predict(x_pred)

plt.plot(channel_profile['distance_km'], channel_profile['elevation_m'], label='Raw Data', color='#1F77B4')
plt.plot(x_pred, y_pred, '-', label='LOWESS', color='#FF7F0E', zorder=6)
plt.xlabel("Distance ($km$)")
plt.ylabel("Elevation ($m$)")
plt.legend(frameon=True)
plt.show()

## Slope Calculation

In [ ]:
x1 = channel_profile['distance_km'][0]*1000
y1 = channel_profile['elevation_m'][0]
xLast = channel_profile['distance_km'].iloc[-1]*1000
yLast = channel_profile['elevation_m'].iloc[-1]
slope = (yLast-y1)/(xLast-x1)
print('Slope:', slope, '| Abs Slope:', abs(slope))

## OLS Regression for All Environment Types (Cross Section Scale)

In [ ]:
env_cols = ['channelBelt', 'wettedChannel', 'barComplex', 'unvegetatedBar', 'vegetatedBar']
for col in env_cols:
    if col in cross_section.columns:
        ols_regression(cross_section, col)

## Example Plot: Water vs Bar Complex

In [ ]:
cols_to_plot = [c for c in ['wettedChannel', 'barComplex'] if c in cross_section.columns]
if cols_to_plot:
    cross_section.plot(x="segment", y=cols_to_plot, kind="bar", stacked=True, color=['#4477aa', '#ee8866'])
    plt.xlabel("% total distance")
    plt.ylabel("Width ($km$)")
    plt.legend(cols_to_plot)
    plt.show()

## Normalised Widths (Percentages)

In [ ]:
if all(col in cross_section.columns for col in ['wettedChannel', 'barComplex']):
    riverCopy_1 = cross_section.copy()
    riverCopy_1[['wettedChannel', 'barComplex']] = riverCopy_1[['wettedChannel', 'barComplex']].apply(lambda x: x/x.sum(), axis=1).multiply(100)
    riverCopy_1.plot(x="segment", y=['wettedChannel','barComplex'], kind="bar", stacked=True, color=['#4477aa', '#ee8866'])
    plt.xlabel("% total distance")
    plt.ylabel("Width normalised ($\%$)")
    plt.legend(['Wetted Channel','Bar Complex'])
    plt.show()

## Additional Analysis: System Scale, Domain Scale, Reach Scale
Repeat similar OLS/statistical analysis for these sheets as needed.

Example for reach scale:

In [ ]:
# For reach_scale, run OLS for each environment column
reach_env_cols = [col for col in reach_scale.columns if col != 'segment' and reach_scale[col].dtype in [np.float64, np.int64]]
for col in reach_env_cols:
    ols_regression(reach_scale, col)

## Example Plot: Reach Scale Environments (Bar Plot)

In [ ]:
if len(reach_env_cols) >= 2:
    reach_scale.plot(x="segment", y=reach_env_cols, kind="bar", stacked=True)
    plt.xlabel("% total distance")
    plt.ylabel("Area or Width")
    plt.legend(reach_env_cols)
    plt.show()